In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout, Bidirectional
import warnings
warnings.filterwarnings('ignore')

def run_prototype():
    print("Starting Model Prototyping...")

    # 1. Load Data
    data_path = os.path.join('data', 'raw', 'SP500_DATASET.CSV')
    if not os.path.exists(data_path):
        print(f"ERROR: Dataset not found at {data_path}.")
        return

    df = pd.read_csv(data_path, parse_dates=['Date'], index_col='Date')
    
    # 2. Engineer Features
    print("Applying features...")
    df['MA_20'] = df['Close'].rolling(window=20).mean()
    df['MA_50'] = df['Close'].rolling(window=50).mean()
    df['Volatility_20'] = df['Close'].rolling(window=20).std()
    
    delta = df['Close'].diff(1)
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    df['RSI_14'] = 100 - (100 / (1 + (gain / loss)))
    
    df.dropna(inplace=True)

    # 3. Scaling
    print("Scaling data...")
    features = ['Open', 'High', 'Low', 'Close', 'Volume', 'MA_20', 'MA_50', 'Volatility_20', 'RSI_14']
    data_filtered = df[features].values
    
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(data_filtered)

    # 4. Sequences (Lookback = 60 days)
    print("Generating sequences...")
    SEQ_LENGTH = 60
    X, y = [], []
    for i in range(SEQ_LENGTH, len(scaled_data)):
        X.append(scaled_data[i-SEQ_LENGTH:i])
        y.append(scaled_data[i, 3]) # Predict 'Close' (index 3)

    X, y = np.array(X), np.array(y)

    # Train/Test Split (80/20)
    split = int(len(X) * 0.8)
    X_train, y_train = X[:split], y[:split]
    X_test, y_test = X[split:], y[split:]
    print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

    # 5. Build Model
    print("Building Bidirectional LSTM Prototype...")
    model = Sequential([
        Bidirectional(LSTM(128, return_sequences=True), input_shape=(X_train.shape[1], X_train.shape[2])),
        Dropout(0.3),
        Bidirectional(LSTM(64, return_sequences=False)),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    model.summary()

    # 6. Train Prototype
    print("Training prototype model (10 epochs)...")
    history = model.fit(
        X_train, y_train, 
        epochs=10, 
        batch_size=32, 
        validation_split=0.1, 
        verbose=1
    )

    # 7. Plot Loss
    print("Plotting training history...")
    plt.figure(figsize=(10, 5))
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title('Prototyping Training & Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Mean Squared Error')
    plt.legend()
    plt.show()

if __name__ == "__main__":
    run_prototype()